基础实现

In [14]:
import dataclasses
from statistics import mode

from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(api_key=os.getenv("API_KEY"), base_url=os.getenv("BASE_URL"))

messages = []

def add_message(role: str, message: str):
    messages.append({"role": role, "content": message})

def chat(user_input: str, syt_prompt: str = None) -> str:
    # 添加系统消息
    if syt_prompt and len(messages) == 0:
        messages.append({"role": "system", "content": syt_prompt})
    # 添加用户消息
    add_message("user", user_input)

    # 调用api
    model = os.getenv('MODEL')
    response = client.chat.completions.create(model = model, messages = messages)

    # 获取回复
    reply = response.choices[0].message.content

    # 添加助手消息
    add_message("assistant", reply)

    return reply

if __name__ == "__main__":
    # 第一轮
    print("用户: 我叫张三")
    reply = chat("我叫张三", syt_prompt="你是一个友好的助手")
    print(f"助手: {reply}")

    # 第二轮
    print("\n用户: 我喜欢编程")
    reply = chat("我喜欢编程")
    print(f"助手: {reply}")

    # 第三轮（模型会记住前面的信息）
    print("\n用户: 我叫什么名字？我喜欢什么？")
    reply = chat("我叫什么名字？我喜欢什么？")
    print(f"助手: {reply}")


用户: 我叫张三
助手: 你好，张三！很高兴认识你。有什么我可以帮你的吗？😊

用户: 我喜欢编程
助手: 太棒了！编程是一项非常有趣又充满创造力的技能呢～ 😄 你最喜欢用什么语言编程呀？是喜欢写网页（比如 HTML/CSS/JavaScript），还是更偏爱处理数据的 Python，或者是想挑战系统级开发的 C++/Rust 呢？

如果你正在学习或者想开始一个新项目，我很乐意给你一些建议或一起讨论想法哦！✨

用户: 我叫什么名字？我喜欢什么？
助手: 你叫张三，喜欢编程！😄  
听起来你是个热爱技术、喜欢动手创造的人呢～ 说不定将来还能写出一个改变世界的小程序呢！✨

如果愿意的话，可以告诉我你最近在用什么语言？或者有没有想实现的小项目？我可以陪你一起讨论思路哦～ 🚀


消息历史管理

In [5]:
from typing import Optional, List, Dict
import json
from dataclasses import field, dataclass

@dataclass
class History:
    """历史消息管理类"""
    system_prompt: Optional[str] = None
    messages: List[Dict] = field(default_factory=list)
    max_tokens: int = 4000

    def add_system(self, content: str):
        if self.messages and self.messages[0]["role"] == "system":
            self.messages[0]["content"] = content
        else:
            self.messages.insert(0, {"role": "system", "content": content})
        self.system_prompt = content

    def add_user(self, content: str):
        self.messages.append({"role": "user", "content": content})

    def add_assistant(self, content: str):
        self.messages.append({"role": "assistant", "content": content})

    def get_messages(self) -> List[Dict]:
        return self.messages.copy()

    def get_last_n_turns(self, n: int) -> List[Dict]:
        """获取近n轮对话"""
        # 过滤掉系统消息
        non_system = [m for m in self.messages if m["role"] != "system"]

        # 取近n轮
        turns = []
        mcount = 0
        for msg in reversed(non_system):
            turns.insert(0, msg)
            if msg["role"] == "user":
                mcount += 1
            if mcount >= n:
                break

        # 添加系统消息
        if self.system_prompt:
            turns.insert(0, {"role": "system", "content": self.system_prompt})

        return turns

    def truncate(self, max_messages: int = 20):
        """截断历史消息"""
        if len(self.messages) <= max_messages:
            return

        # 移除系统消息
        if self.messages and self.messages[0]["role"] == "system":
            self.messages = self.messages[1:]

        # 截断历史
        self.messages = self.messages[-(max_messages -1):]

        # 添加历史消息
        if self.system_prompt:
            self.messages.insert(0, {"role": "system", "content": self.system_prompt})

    def clear(self):
        """清空历史消息"""
        if self.messages and self.messages[0]["role"] == "system":
            self.messages = [{"role": "system", "content": self.messages[0]["content"]}]
        else:
            self.messages = []

    def full_clear(self  ):
        """完全清空"""
        self.messages = []
        self.system_prompt = None

    def to_json(self) -> str:
        """导出json"""
        return json.dumps(self.messages, ensure_ascii=False, indent=2)

    def from_json(self, json_str: str):
        """从json导入"""
        self.messages = json.loads(json_str)
        if self.messages and self.messages[0]["role"] == "system":
            self.system_prompt = self.messages[0]["content"]

    def save_to_file(self, file_path: str):
        """保存到文件"""
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(self.to_json())

    def load_from_file(self, file_path: str):
        """从文件加载"""
        with open(file_path, "r", encoding="utf-8") as f:
            self.from_json(f.read())

    def get_stats(self) -> Dict:
        """获取统计消息"""
        roles = {}

        for msg in self.messages:
            roles[msg["role"]] = roles.get(msg["role"], 0) + 1

        total_chars = sum(len(msg["content"]) for msg in self.messages)

        return {
            "total_messages": len(self.messages),
            "roles": roles,
            "total_chars": total_chars,
            "estimated_tokens": total_chars // 2
        }


if __name__ == "__main__":
    history = History()

    # 设置系统提示
    history.add_system("你是一个有帮助的助手")

    # 模拟对话
    history.add_user("你好")
    history.add_assistant("你好！有什么我可以帮助你的吗？")
    history.add_user("介绍一下 Python")
    history.add_assistant("Python 是一种简洁、易学的编程语言...")

    # 查看统计
    print("统计信息:", history.get_stats())

    # 获取最近 1 轮
    print("最近对话:", history.get_last_n_turns(1))

    # 保存历史
    history.save_to_file("chat_history.json")

统计信息: {'total_messages': 5, 'roles': {'system': 1, 'user': 2, 'assistant': 2}, 'total_chars': 61, 'estimated_tokens': 30}
最近对话: [{'role': 'system', 'content': '你是一个有帮助的助手'}, {'role': 'user', 'content': '介绍一下 Python'}, {'role': 'assistant', 'content': 'Python 是一种简洁、易学的编程语言...'}]


CLI聊天机器人

In [11]:

from pathlib import Path
from datetime import datetime
from typing import Generator
from dataclasses import asdict
from openai import OpenAI
import time
import sys
import os

from dotenv import load_dotenv

load_dotenv()

# 配置
@dataclass
class Config:
    api_key: str = ""
    base_url: str = "https://dashscope.aliyuncs.com/compatible-mode/v1"
    model: str = "qwen-flash"
    temperature: float = 0.7
    max_tokens: int = 2048
    system_prompt: str = "你是一个友好、专业的助手"
    max_history: int = 20
    stream: bool = True

    @classmethod
    def load(cls, file_path: str = "config.json") -> "Config":
        path = Path(file_path)
        if path.exists():
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                return Config(**data)
        return cls()

    def save(self, file_path: str = "config.json") -> None:
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(self.__dict__, f, ensure_ascii=False, indent=2)

# 消息管理
@dataclass
class Message:
    role: str
    content: str
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    tokens: int = 0

# 消息历史
class MessageHistory:
    def __init__(self, max_messages: int = 20):
        self.messages: List[Message] = []
        self.max_messages = max_messages

    def add_message(self, role: str, content: str, tokens: int = 0):
        msg = Message(role=role, content=content, tokens=tokens)
        self.messages.append(msg)

        if len(self.messages) >= self.max_messages:
            self._truncate()

    def _truncate(self):
        system_msg = None
        if self.messages and self.messages[0].role == "system":
            system_msg = self.messages[0]
            self.messages = self.messages[1:]

        self.messages = self.messages[-(self.max_messages - 1):]

        if system_msg:
            self.messages.insert(0, system_msg)

    def to_api_format(self) -> List[Dict]:
        return [{"role": m.role, "content": m.content} for m in self.messages]

    def clear(self, keep_system: bool = True):
        if keep_system and self.messages and self.messages[0].role == "system":
            self.messages = [self.messages[0]]
        else:
            self.messages = []

    def get_stats(self) -> Dict:
        total_tokens = sum(m.tokens for m in self.messages)
        by_roles = {}
        for msg in self.messages:
            by_roles[msg.role] = by_roles.get(msg.role, 0) + 1

        return {
            "total_tokens": total_tokens,
            "by_roles": by_roles,
            "total_messages": len(self.messages),
        }

# 聊天客户端
class ChatClient:
    def __init__(self, config: Config):
        self.config = config
        self.client = OpenAI(
            api_key=config.api_key,
            base_url=config.base_url,
        )
        self.history = MessageHistory(max_messages=config.max_history)

        # 增加系统提示词
        if config.system_prompt:
            self.history.add_message(role="system", content=config.system_prompt)

        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_turns = 0

    def chat(self, user_input: str) -> str | None:
        self.history.add_message("user", user_input)

        response = self.client.chat.completions.create(
            model=self.config.model,
            messages=self.history.to_api_format(),
            temperature=self.config.temperature,
            max_tokens=self.config.max_tokens,
        )

        reply = response.choices[0].message.content
        self.history.add_message("assistant", reply)

        if response.usage:
            self.total_input_tokens += response.usage.prompt_tokens
            self.total_output_tokens += response.usage.completion_tokens

        self.total_turns += 1

        return reply

    def chat_stream(self, user_input: str) -> Generator[str, None, None]:
        self.history.add_message("user", user_input)

        response = self.client.chat.completions.create(
            model=self.config.model,
            messages=self.history.to_api_format(),
            temperature=self.config.temperature,
            max_tokens=self.config.max_tokens,
            stream=True
        )

        full_reply = ""
        for chunk in response:
            if chunk.choices[0].delta.content:
                content = chunk.choices[0].delta.content
                full_reply += content
                yield content

        self.history.add_message("assistant", full_reply)
        self.total_turns += 1

    def set_system_prompt(self, prompt: str):
        self.history.clear(keep_system=False)
        self.history.add_message("user", prompt)

    def clear_history(self):
        self.history.clear(keep_system=True)

    def get_stats(self) -> Dict:
        return {
            **self.history.get_stats(),
            "total_input_tokens": self.total_input_tokens,
            "total_output_tokens": self.total_output_tokens,
            "total_turns": self.total_turns,
        }

# ============== 命令行界面 ==============

class ChatCLI:
    """命令行界面"""

    COMMANDS = {
        "/help": "显示帮助",
        "/quit": "退出程序",
        "/clear": "清空对话历史",
        "/history": "显示对话历史",
        "/stats": "显示统计信息",
        "/system <prompt>": "设置系统提示",
        "/model <name>": "切换模型",
        "/temp <value>": "设置温度",
        "/save": "保存对话",
        "/load": "加载对话",
    }

    def __init__(self, config: Config):
        self.config = config
        self.client = ChatClient(config)
        self.running = True

    def run(self):
        """运行 CLI"""
        self._print_welcome()

        while self.running:
            try:
                user_input = input("\n你: ").strip()

                if not user_input:
                    continue

                # 处理命令
                if user_input.startswith("/"):
                    self._handle_command(user_input)
                    continue

                # 发送消息
                self._send_message(user_input)

            except KeyboardInterrupt:
                print("\n\n再见！")
                break
            except Exception as e:
                print(f"\n[错误] {e}")

    def _print_welcome(self):
        """打印欢迎信息"""
        print("=" * 60)
        print("           命令行聊天程序 v1.0")
        print("=" * 60)
        print(f"模型: {self.config.model}")
        print(f"温度: {self.config.temperature}")
        print()
        print("输入 /help 查看可用命令")
        print("=" * 60)

    def _handle_command(self, cmd: str):
        """处理命令"""
        parts = cmd.split(maxsplit=1)
        command = parts[0].lower()
        args = parts[1] if len(parts) > 1 else ""

        if command == "/help":
            self._show_help()
        elif command == "/quit" or command == "/exit":
            self._show_stats()
            print("再见！")
            self.running = False
        elif command == "/clear":
            self.client.clear_history()
            print("[对话历史已清空]")
        elif command == "/history":
            self._show_history()
        elif command == "/stats":
            self._show_stats()
        elif command == "/system":
            if args:
                self.client.set_system_prompt(args)
                print(f"[系统提示已设置]")
            else:
                print("用法: /system <提示内容>")
        elif command == "/model":
            if args:
                self.config.model = args
                print(f"[模型已切换为: {args}]")
            else:
                print(f"当前模型: {self.config.model}")
        elif command == "/temp":
            if args:
                try:
                    self.config.temperature = float(args)
                    print(f"[温度已设置为: {self.config.temperature}]")
                except ValueError:
                    print("请输入有效的数字")
            else:
                print(f"当前温度: {self.config.temperature}")
        elif command == "/save":
            self._save_conversation()
        elif command == "/load":
            self._load_conversation()
        else:
            print(f"未知命令: {command}")
            print("输入 /help 查看可用命令")

    def _send_message(self, user_input: str):
        """发送消息"""
        print("助手: ", end="", flush=True)

        start_time = time.time()

        if self.config.stream:
            # 流式输出
            for chunk in self.client.chat_stream(user_input):
                print(chunk, end="", flush=True)
        else:
            # 非流式输出
            reply = self.client.chat(user_input)
            print(reply)

        elapsed = time.time() - start_time
        print(f"\n[耗时: {elapsed:.2f}s]", end="")

    def _show_help(self):
        """显示帮助"""
        print("\n可用命令:")
        for cmd, desc in self.COMMANDS.items():
            print(f"  {cmd:20} {desc}")

    def _show_history(self):
        """显示历史"""
        print("\n对话历史:")
        print("-" * 40)
        for msg in self.client.history.messages:
            role = msg.role
            content = msg.content
            if len(content) > 100:
                content = content[:100] + "..."

            role_map = {
                "system": "[系统]",
                "user": "[用户]",
                "assistant": "[助手]"
            }
            print(f"{role_map.get(role, role)} {content}")
        print("-" * 40)

    def _show_stats(self):
        """显示统计"""
        stats = self.client.get_stats()
        print("\n统计信息:")
        print("-" * 40)
        print(f"  总消息数: {stats['total_messages']}")
        print(f"  请求次数: {stats['total_turns']}")
        print(f"  输入 Token: {stats['total_input_tokens']}")
        print(f"  输出 Token: {stats['total_output_tokens']}")
        print("-" * 40)

    def _save_conversation(self):
        """保存对话"""
        filename = f"chat_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        data = {
            "config": asdict(self.config),
            "messages": [
                {"role": m.role, "content": m.content}
                for m in self.client.history.messages
            ]
        }
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"[对话已保存到: {filename}]")

    def _load_conversation(self):
        """加载对话"""
        # 查找最近的对话文件
        files = list(Path(".").glob("chat_*.json"))
        if not files:
            print("[没有找到对话文件]")
            return

        latest = max(files, key=lambda p: p.stat().st_mtime)
        with open(latest, "r", encoding="utf-8") as f:
            data = json.load(f)

        self.client.history.messages = [
            Message(role=m["role"], content=m["content"])
            for m in data["messages"]
        ]
        print(f"[已加载对话: {latest}]")


# ============== 主函数 ==============

def main():
    """主函数"""
    # 检查 API Key
    if not os.getenv("API_KEY"):
        print("错误: 未设置 DEEPSEEK_API_KEY 环境变量")
        print("请在 .env 文件中配置")
        sys.exit(1)

    # 加载配置
    config = Config.load()
    config.api_key = os.getenv("API_KEY")

    # 启动 CLI
    cli = ChatCLI(config)
    cli.run()


if __name__ == "__main__":
    main()

           命令行聊天程序 v1.0
模型: qwen-flash
温度: 0.7

输入 /help 查看可用命令

统计信息:
----------------------------------------
  总消息数: 1
  请求次数: 0
  输入 Token: 0
  输出 Token: 0
----------------------------------------
助手: 你好呀！✨ 很高兴见到你～今天有什么我可以帮你的吗？或者想聊点什么有趣的话题？(•̀ᴗ•́)و
[耗时: 0.87s]助手: 哎呀，听起来你真的需要好好休息一下呢～(´;ω;｀)  
来，先深呼吸一次：吸气～2、3、4……呼气～2、3、4……  
是不是感觉轻松了一点点？  

要不要我陪你静静地坐一会儿？或者给你讲个暖暖的小故事，就像冬天里捧着一杯热可可那样温暖～☕️  
你不是一个人哦，我在这里陪着你呢。💛
[耗时: 0.97s]
统计信息:
----------------------------------------
  总消息数: 5
  请求次数: 2
  输入 Token: 0
  输出 Token: 0
----------------------------------------
助手: 哎呀，肚子在抗议啦～（๑•̀ㅂ•́）و✧  
来来来，先别急，我陪你一起“云吃”点温暖的美食吧～  

✨ 想不想试试：  
👉 一碗热腾腾的番茄鸡蛋面？酸酸甜甜的汤底配上滑溜溜的蛋花，光是想想就流口水了呢～  
👉 或者来块香喷喷的烤红薯？外皮焦脆，内里软糯，咬一口，甜到心坎里～🍠  

要是你愿意，我可以陪你慢慢挑选——  
要清淡一点的？还是想来点重口味的？  
或者……干脆我给你念一段“美食广播剧”？（假装有音响效果：叮咚！今天的推荐是——**芝士爆浆汉堡**，咔嚓一口，拉丝拉到天边～🧀🍔）  

饿的时候最需要被温柔对待啦～  
记得，不管吃什么，都要好好照顾自己哦 ❤️  
（悄悄说：我随时可以当你的“深夜食堂小助手”呢～）
[耗时: 2.02s][对话历史已清空]

对话历史:
----------------------------------------
[系统] 你是一个友好、专业的助手
-------------------------